In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv('data//1_preprocessed//preprocessed_dataset_with_clusters.csv')
genre_to_cluster = pd.read_csv('data//1_preprocessed//genre_to_cluster_mapping.csv', index_col=0, squeeze=True).to_dict()

y = data.genre_clusters
X = data[['duration_ms','explicit','danceability','energy','key','loudness','mode','speechiness','acousticness','instrumentalness','liveness','valence','tempo','time_signature']]

# Stratified split to maintain class distribution in train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

# Categorical features (will want to one-hot encode these): 'key', 'mode', 'time_signature'
categorical_cols = ['key', 'mode', 'time_signature']
print("Cardinality of categorical columns:")
for col in categorical_cols:
    print(f"{col}: {X[col].nunique()}")
# Only key (12) may have too high of a cardinality, especially since it's already likely not a very important feature, so we'll drop it.
categorical_cols.remove('key')

numerical_cols = ['duration_ms','explicit','danceability','energy','loudness','speechiness','acousticness','instrumentalness','liveness','valence','tempo']
print("Numerical columns:", numerical_cols)

Cardinality of categorical columns:
key: 12
mode: 2
time_signature: 5
Numerical columns: ['duration_ms', 'explicit', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

class EncodedTargetPipeline(Pipeline):
    def fit(self, X, y=None, **fit_params):
        self.label_encoder_ = LabelEncoder()
        encoded_y = self.label_encoder_.fit_transform(y) if y is not None else y
        return super().fit(X, encoded_y, **fit_params)

    def predict(self, X, **predict_params):
        encoded_preds = super().predict(X, **predict_params)
        return self.label_encoder_.inverse_transform(encoded_preds)

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough'  # Keep the numerical columns as they are; already preprocessed
)

In [3]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=0, verbose=True)
my_pipeline = EncodedTargetPipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

my_pipeline.fit(X_train, y_train)

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:   20.8s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:   54.1s finished


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given strin

In [10]:
from sklearn.metrics import classification_report

train_preds = my_pipeline.predict(X_train)
print("Training Accuracy:", (train_preds == y_train).mean())
train_report = classification_report(y_train, train_preds, output_dict=True)
df_train_report = pd.DataFrame(train_report).T
print(df_train_report.sort_values("f1-score").head(20))  # worst performing genres in training set

test_preds = my_pipeline.predict(X_test)
print("Testing Accuracy:", (test_preds == y_test).mean())
test_report = classification_report(y_test, test_preds, output_dict=True)
df_test_report = pd.DataFrame(test_report).T
print(df_test_report.sort_values("f1-score").head(20))  # worst performing genres in test set

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    6.2s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:   17.1s finished


Training Accuracy: 0.8172921322339524
                   precision    recall  f1-score  support
reggae              0.405437  0.428750  0.416768    800.0
reggaeton           0.425629  0.465000  0.444444    800.0
singer-songwriter   0.484252  0.461250  0.472471    800.0
latino              0.444201  0.510692  0.475132    795.0
songwriter          0.454139  0.507500  0.479339    800.0
latin               0.543307  0.435606  0.483532    792.0
edm                 0.485643  0.489308  0.487469    795.0
alternative         0.510811  0.473091  0.491228    799.0
indie               0.524544  0.468672  0.495036    798.0
alt-rock            0.515789  0.490613  0.502886    799.0
house               0.552342  0.501877  0.525902    799.0
metal               0.530778  0.571965  0.550602    799.0
rock                0.606310  0.552500  0.578156    800.0
punk                0.582927  0.598248  0.590488    799.0
indie-pop           0.597701  0.585000  0.591282    800.0
punk-rock           0.614232  0.61

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    4.6s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    5.8s finished


Testing Accuracy: 0.25420519594892116
                   precision    recall  f1-score  support
brazil              0.014815  0.010000  0.011940    200.0
alt-rock            0.017964  0.015000  0.016349    200.0
punk-rock           0.022222  0.020000  0.021053    200.0
electronic          0.062500  0.025000  0.035714    200.0
singer-songwriter   0.038278  0.040000  0.039120    200.0
punk                0.046154  0.045000  0.045570    200.0
songwriter          0.045643  0.055000  0.049887    200.0
groove              0.066667  0.040000  0.050000    200.0
j-rock              0.055249  0.050000  0.052493    200.0
mpb                 0.055866  0.050000  0.052770    200.0
metal               0.055838  0.055276  0.055556    199.0
hard-rock           0.065359  0.050000  0.056657    200.0
edm                 0.065990  0.065657  0.065823    198.0
psych-rock          0.085106  0.060302  0.070588    199.0
reggae              0.075000  0.075000  0.075000    200.0
dub                 0.075000  0.07

In [11]:
from xgboost import XGBClassifier

model = XGBClassifier(random_state=0, verbose=True)
my_pipeline = EncodedTargetPipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

my_pipeline.fit(X_train, y_train)

c:\Users\logan\miniconda3\envs\datasci\Lib\site-packages\xgboost\training.py:200: UserWarning: [10:30:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given strin

In [12]:
train_preds = my_pipeline.predict(X_train)
print("Training Accuracy:", (train_preds == y_train).mean())
train_report = classification_report(y_train, train_preds, output_dict=True)
df_train_report = pd.DataFrame(train_report).T
print(df_train_report.sort_values("f1-score").head(20))  # worst performing genres in training set

test_preds = my_pipeline.predict(X_test)
print("Testing Accuracy:", (test_preds == y_test).mean())
test_report = classification_report(y_test, test_preds, output_dict=True)
df_test_report = pd.DataFrame(test_report).T
print(df_test_report.sort_values("f1-score").head(20))  # worst performing genres in test set

Training Accuracy: 0.723125529783463
                   precision    recall  f1-score  support
reggae              0.382889  0.341250  0.360872    800.0
songwriter          0.415775  0.388750  0.401809    800.0
alternative         0.565315  0.314143  0.403862    799.0
alt-rock            0.474747  0.352941  0.404882    799.0
singer-songwriter   0.420000  0.420000  0.420000    800.0
reggaeton           0.378456  0.496250  0.429421    800.0
latin               0.504202  0.378788  0.432588    792.0
indie               0.501603  0.392231  0.440225    798.0
latino              0.381425  0.532075  0.444328    795.0
edm                 0.406406  0.510692  0.452620    795.0
house               0.470960  0.466834  0.468887    799.0
metal               0.505420  0.466834  0.485361    799.0
indie-pop           0.570248  0.431250  0.491103    800.0
punk                0.552478  0.474343  0.510438    799.0
rock                0.583333  0.455000  0.511236    800.0
brazil              0.669903  0.432